# Crime classifier trained from scratch
Run notebook 01 first and review labels, annotation schemas, readability, and source IDs. Select Runtime → Change runtime type → T4 GPU, then run cells in order. This notebook defaults to preparing the balanced prototype without training. Full training requires your separate approval. Checkpoints and metrics are saved each epoch to Drive; a disconnected in-progress epoch may be lost. Rerunning training starts fresh from the fixed seed; existing best checkpoints can be evaluated without retraining.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/AI-Crime-Investigation-FYP')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ranaalihaider99993-source/AI-Crime-Investigation-FYP.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
from scripts.common import seed_all
seed_all(42)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/AI-Crime-Investigation-FYP')
except ImportError:
    OUTPUT = REPO / 'artifacts'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Persistent output:', OUTPUT)


In [ ]:
import kagglehub
try:
    DATA = Path(kagglehub.dataset_download('mateohervas/dcsass-dataset'))
except Exception as exc:
    raise RuntimeError('Kaggle download failed. Check network/access; if authentication is required, use Colab secrets or kagglehub.login(), never commit credentials.') from exc
print('Dataset root:', DATA)


In [ ]:
from scripts.data import inspect, split_manifest
from pathlib import Path
import json
CONFIG = json.loads(Path('configs/prototype.json').read_text())
# First real-data smoke only: 3 original sources/class, one clip/source, 2 epochs.
CONFIG.update(sources_per_class=3, epochs=2, batch_size=4)
# Do not enlarge this experiment until the smoke succeeds and full training is approved.
# Rebuild paths for this runtime. Review the new manifest BEFORE enabling training.
print(inspect(DATA, OUTPUT / 'inspection'))
print(CONFIG)


Confirm that all retained clips have the intended class, source IDs identify original videos, and each source has only one class. Flat unknown clip naming needs a human-reviewed source mapping in manifest.csv. Do not mark this review complete based on class names alone. This baseline averages logits from uniformly sampled frames; it does not localize events or model motion. Clip category labels may include normal frames.

In [ ]:
SOURCE_MAPPING_REVIEWED = False  # Set True only after reviewing actual manifest/source grouping.
RUN_PROTOTYPE_TRAINING = False  # Enable for a small prototype, not full training.
if SOURCE_MAPPING_REVIEWED:
    splits = split_manifest(OUTPUT / 'inspection/manifest.csv', OUTPUT / 'classifier_splits.csv',
                            seed=CONFIG['seed'], per_class=CONFIG['sources_per_class'])
    print('Selected independent source clips:', len(splits))
if RUN_PROTOTYPE_TRAINING:
    if not SOURCE_MAPPING_REVIEWED:
        raise ValueError('Review source grouping before training')
    from scripts.classifier import train
    history = train(OUTPUT / 'classifier_splits.csv', OUTPUT / 'classifier', CONFIG)
else:
    print('Training disabled. Review source grouping, then enable the prototype training flag.')


In [ ]:
# Evaluate only after this run trained successfully; never report an old checkpoint as a new run.
if RUN_PROTOTYPE_TRAINING:
    from scripts.classifier import evaluate
    metrics = evaluate(OUTPUT / 'classifier_splits.csv',
                       OUTPUT / 'classifier/crime_classifier.pth',
                       OUTPUT / 'classifier/test_metrics.json')
    print(json.dumps(metrics, indent=2))
    print('Best checkpoint:', OUTPUT / 'classifier/crime_classifier.pth')
    print('This 15-clip smoke experiment is not a reliable performance benchmark.')
    # The checkpoint includes state_dict, config and class order for later Django loading.
    DOWNLOAD_CHECKPOINT = False  # Set True to download a copy to your computer.
    if DOWNLOAD_CHECKPOINT:
        from google.colab import files
        files.download(str(OUTPUT / 'classifier/crime_classifier.pth'))
